In [1]:
import pandas as pd
import ast
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# load data
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

#menggabungkan kedua dataset beserta judul
movies = movies.merge(credits, on='title')

In [2]:
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

# membersihkan genre dan keywords
movies['genre'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

# castnya diambil 3 pemain saja
def convert_cast(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            L.append(i['name'])
            counter += 1
        else :
            break
    return L

movies['cast'] = movies['cast'].apply(convert_cast)

In [3]:
# hilangkan spasi pada nama 
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ","") for i in x])
movies['genres'] = movies['genres'].apply(lambda x: [i.replace(" ","") for i in x])

# gabungin semuanya jadi satu kolom 'tags'
movies['tags'] = movies['overview'].astype(str) + " " + \
                movies['genres'].apply(lambda x: " ".join(x)) + " " + \
                movies['cast'].apply(lambda x: " ".join(x))

new_df = movies[['movie_id', 'title','tags']].copy()

#memastikan data kosong(NaN) diisi dengan string kosong
new_df['tags'] = new_df['tags'].fillna('')

#menggunakan str(x) untuk memaksa data menjadi string sebelum di lower
new_df['tags'] = new_df['tags'].apply(lambda x: str(x).lower())

In [4]:
#vektorisasi
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
vector = tfidf.fit_transform(new_df['tags']).toarray()

#menghitung kemiripan
similarity = cosine_similarity(vector)

# export untuk flask
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))

print("Berhasil memproses 5000 film")

Berhasil memproses 5000 film


In [5]:
def recommend(movie_title):
    # ceek apakah film yang dicari ada di dalam dataset
    if movie_title not in new_df['title'].values:
        print(f"Maaf, film '{movie_title}' tidak ditemukan di dataset.")
        return
        
    # mencari posisi baris film yang dicari
    movie_index = new_df[new_df['title'] == movie_title].index[0]
    
    # mengambil daftar skor kemiripan film 
    distances = similarity[movie_index]
    
    # mengambil 5 film teratas
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    print(f"Jika kamu suka '{movie_title}', coba tonton ini:")
    for i in movies_list:
        print(f"- {new_df.iloc[i[0]]['title']}")

# testing
recommend('Batman Begins')

Jika kamu suka 'Batman Begins', coba tonton ini:
- The Dark Knight Rises
- Batman
- The Dark Knight
- Batman Returns
- Batman
